# Week 8: Find Structure, Don't Fabricate It

**Challenge:** Discover the hidden structure of mental distress using PCA and clustering on the DASS-42 dataset.

**Two phases:**
1. **Dimensionality reduction** — PCA on 42 DASS items + UMAP visualisation
2. **Clustering** — k-means on personality + distress profiles, with stability checks

**What makes this different from Weeks 2–6:** There's no target variable. No "right answer." You're exploring structure, and the challenge is figuring out whether what you find is real or noise.

---

**Cells 1–5 are done for you.** Run them to load, clean, and score the data. Your AI helps you fill in cells 6 onwards.

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

print('All imports loaded successfully!')

In [ ]:
# Cell 2: Load the data
# Same dataset as Week 4 — tab-separated
data = pd.read_csv('data/dass42_data.csv', sep='\t')

print(f'Dataset loaded: {data.shape[0]:,} respondents, {data.shape[1]} variables')
print(f'First few columns: {list(data.columns[:10])}')

In [ ]:
# Cell 3: Data quality filtering

# 1. Remove unrealistic ages (some are birth years like 1998)
data = data[(data['age'] >= 10) & (data['age'] <= 100)]
print(f'After age filter (10-100): {len(data):,} rows')

# 2. Remove careless responders
# VCL6, VCL9, VCL12 are fake vocabulary words that don't exist.
# If someone claims to know them (value = 1), they weren't paying attention.
data = data[(data['VCL6'] == 0) & (data['VCL9'] == 0) & (data['VCL12'] == 0)]
print(f'After removing careless responders: {len(data):,} rows')

In [ ]:
# Cell 4: Score DASS subscales
# DASS-42: 42 items coded 1-4. Recode to 0-3 (subtract 1), then sum per subscale.

dass_cols = [f'Q{i}A' for i in range(1, 43)]
for col in dass_cols:
    data[col] = data[col] - 1  # Recode 1-4 to 0-3

# Depression: 14 items
dep_items = [f'Q{i}A' for i in [3, 5, 10, 13, 16, 17, 21, 24, 26, 31, 34, 37, 38, 42]]
data['DASS_Depression'] = data[dep_items].sum(axis=1)

# Anxiety: 14 items
anx_items = [f'Q{i}A' for i in [2, 4, 7, 9, 15, 19, 20, 23, 25, 28, 30, 36, 40, 41]]
data['DASS_Anxiety'] = data[anx_items].sum(axis=1)

# Stress: 14 items
str_items = [f'Q{i}A' for i in [1, 6, 8, 11, 12, 14, 18, 22, 27, 29, 32, 33, 35, 39]]
data['DASS_Stress'] = data[str_items].sum(axis=1)

print('DASS subscales scored:')
print(f'  Depression: M={data["DASS_Depression"].mean():.1f}, SD={data["DASS_Depression"].std():.1f}')
print(f'  Anxiety:    M={data["DASS_Anxiety"].mean():.1f}, SD={data["DASS_Anxiety"].std():.1f}')
print(f'  Stress:     M={data["DASS_Stress"].mean():.1f}, SD={data["DASS_Stress"].std():.1f}')

In [ ]:
# Cell 5: Score Big Five personality from TIPI
# TIPI: 10 items, 2 per trait. Reverse-score items 2, 4, 6, 8, 10.
# Same scoring approach as Week 4: reverse = 8 - original

for i in [2, 4, 6, 8, 10]:
    data[f'TIPI{i}R'] = 8 - data[f'TIPI{i}']

data['Extraversion'] = (data['TIPI1'] + data['TIPI6R']) / 2
data['Agreeableness'] = (data['TIPI2R'] + data['TIPI7']) / 2
data['Conscientiousness'] = (data['TIPI3'] + data['TIPI8R']) / 2
data['EmotionalStability'] = (data['TIPI4R'] + data['TIPI9']) / 2
data['Openness'] = (data['TIPI5'] + data['TIPI10R']) / 2

big5_traits = ['Extraversion', 'Agreeableness', 'Conscientiousness',
               'EmotionalStability', 'Openness']

print('Big Five scored:')
for trait in big5_traits:
    print(f'  {trait}: M={data[trait].mean():.2f}, SD={data[trait].std():.2f}')

print()
print(f'Your data is ready: {len(data):,} respondents')
print(f'  DASS items (for PCA): {len(dass_cols)} columns')
print(f'  Clustering features: 5 Big Five + 3 DASS subscales = 8')

## Summary so far

You now have:
- `data` — a DataFrame with ~34,500 respondents (after data quality filtering)
- `dass_cols` — a list of the 42 DASS item column names (for PCA)
- `DASS_Depression`, `DASS_Anxiety`, `DASS_Stress` — subscale totals (0–42 each)
- `Extraversion`, `Agreeableness`, `Conscientiousness`, `EmotionalStability`, `Openness` — Big Five scores

**From here, your AI helps you build the analysis.** The cells below give you the structure — ask your AI to fill in the code.

---

## Step 1: Plan Your Analysis

Before writing any code, ask your AI to create an analysis plan. Use the planning prompt from the challenge brief — or adapt it to your own approach.

Paste your plan in the cell below.

*Your analysis plan here...*

## Step 2: PCA on 42 DASS Items

Standardise the 42 DASS items and fit PCA. Create a scree plot showing variance explained per component.

Ask your AI: *"Standardise the 42 DASS columns in `dass_cols` using StandardScaler, fit PCA with all components, and create a scree plot showing variance explained by each component with a cumulative line."*

In [ ]:
# Cell 6: PCA — standardise, fit, scree plot
# YOUR CODE HERE — ask your AI to fill this in

## Step 3: Examine PCA Loadings

What do the top components represent? Which DASS items load highest on each? Do the loadings match the Depression/Anxiety/Stress subscale structure?

Ask your AI: *"Show the top 5 item loadings (positive and negative) for the first 3 PCA components. Display as a heatmap or table."*

In [ ]:
# Cell 7: PCA loadings
# YOUR CODE HERE

## Step 4: UMAP Visualisation

Create a 2D projection of participants using UMAP, coloured by depression severity. Remember: distances between clusters in UMAP are **not** meaningful — only local structure is preserved.

Ask your AI: *"Fit UMAP (n_components=2, random_state=42) on the standardised DASS items and create a scatter plot coloured by DASS_Depression."*

**Note:** If UMAP isn't installed, run `pip install umap-learn` in your terminal first.

In [ ]:
# Cell 8: UMAP visualisation
# YOUR CODE HERE

## Step 5: Cluster Participants

Run k-means clustering on standardised Big Five + DASS subscales (8 features). Try at least k=2, 3, 4, and 5. Use silhouette scores and the elbow method to evaluate.

Ask your AI: *"Run k-means for k=2,3,4,5 on the 8 standardised features (5 Big Five + 3 DASS subscales). Plot an elbow curve (inertia vs k) and silhouette scores vs k."*

In [ ]:
# Cell 9: k-means clustering — try multiple k values
# YOUR CODE HERE

## Step 6: Profile Your Clusters

For your chosen k, describe each cluster psychologically. What are the mean values of each feature? Don't just report numbers — interpret what each cluster *means* in psychological terms.

Ask your AI: *"For my k-means solution with k=[your best k], create a bar chart showing the mean of each feature (Big Five + DASS subscales) per cluster. Label the clusters descriptively."*

In [ ]:
# Cell 10: Cluster profiling
# YOUR CODE HERE

## Step 7: Stability Checks

Rerun k-means with at least 3 different random seeds. Do the same people end up in the same clusters? Report the percentage of participants who change cluster across runs.

Ask your AI: *"Rerun k-means with random_state=0, 42, 123 and compare cluster assignments. What percentage of participants change cluster?"*

In [ ]:
# Cell 11: Stability checks
# YOUR CODE HERE

## Step 8: Write a Methods Paragraph

This is the documentation skill. Ask your AI to write a methods paragraph describing your analysis — then **verify every detail**. Use the documentation prompt from the challenge brief, filling in your actual numbers.

Paste the AI's draft below, then **highlight in bold** any changes you had to make.

Ask your AI: *"Write a methods paragraph for a psychology journal (APA style) describing my PCA and clustering analysis. Include: sample size, measures, preprocessing, number of components, variance explained, clustering algorithm, number of clusters, evaluation metrics. Be specific about software (Python, scikit-learn)."*

*Your methods paragraph here...
(Bold any parts you corrected from the AI's draft)*

## Save Your Results

Save any figures you want to include in your presentation.

In [ ]:
# Cell 12: Save figures
# Uncomment when ready:
# fig.savefig('scree_plot.png', dpi=300, bbox_inches='tight')
# print('Figure saved!')

## Bonus Challenges

Finished early? Try one of these:

1. **Parallel analysis** — statistically principled way to choose the number of components
2. **Varimax rotation** — rotate the PCA solution for cleaner loadings
3. **Hierarchical clustering** — build a dendrogram and compare to k-means
4. **DBSCAN** — try density-based clustering with automatic noise detection
5. **Cluster by demographics** — do clusters differ by age, gender, or country?
6. **Bootstrap stability** — resample 100 times and measure co-assignment rates
7. **Compare PCA and UMAP** — same data in both projections side by side
8. **Try MALD/ELP** — apply the pipeline to a psycholinguistic dataset

See the challenge brief (README.md) for details on each.